# Generator viewer

**One generator, one sample, in detail.** Pick a generator and its parameters
in the parameters cell, draw a single sample, and look at it. Change `SEED` and
re-run to see how the structure varies from draw to draw.

For a quick look at what every generator produces, use
`generators_overview.ipynb` instead.

- 2D samples are shown as an image.
- 3D samples are shown as an interactive surface of one phase (drag to rotate),
  plus three mid-plane slices.

The interactive 3D view uses [Plotly](https://plotly.com/python/)
(`pip install plotly`). Without it, the notebook falls back to a static
Matplotlib view.

In [ ]:
import itertools
import time

import matplotlib.pyplot as plt
import numpy as np
from skimage import measure

from compgen import gen

try:
    import plotly.graph_objects as go
except ImportError:
    go = None
    print("Plotly not found: 3D views will be static. Install it with `pip install plotly`.")

## Parameters

`SEED` fixes the random draw: the same seed and parameters always give the same
structure. Any generator from `compgen.gen` works here; a few alternatives are
commented out below.

In [ ]:
SEED = 0   # change to explore randomness
DIM = 3    # 2 or 3
N = 128     # voxels per side

generator = gen.VoronoiGenerator(
    [N] * DIM,
    grain_size=0.65,   # fraction of the unit cell; n_grains = round(grain_size**-DIM)
    thickness=0.1,    # wall width (or strut diameter) as a fraction of grain_size
    lloyd_iter=2,      # > 0 gives more equiaxed grains
    mode="edges",      # 3D only: "faces" (walls) or "edges" (struts)
    invert=False,      # False: grains True, boundaries False
    random_seed=SEED,
)

# generator = gen.SpinodalGenerator([N] * DIM, vf=0.4, l=N / 3, bandwidth=3, random_seed=SEED)
# generator = gen.ParticleGenerator([N] * DIM, vf=0.2, kind="spheres" if DIM == 3 else "disks",
#                                   shape_params={"size_m": 0.15}, random_seed=SEED)
# generator = gen.RandomGenerator([N] * DIM, vf=0.3, random_seed=SEED)

## Sample

In [ ]:
start = time.perf_counter()
structure = generator.sample()
elapsed = time.perf_counter() - start

print(f"{type(generator).__name__}: shape {structure.shape}, dtype {structure.dtype}")
print(f"True fraction {structure.mean():.3f}, False fraction {1 - structure.mean():.3f}")
print(f"generated in {elapsed:.2f} s")

## View

`PHASE` picks the phase drawn as a surface in 3D (`None`: the minority phase,
e.g. the boundaries of a Voronoi structure with `invert=False`). For large `N`,
raise `STEP` to coarsen the surface and keep the view responsive.

Unit-cell edges are outlined in every view. Set `TILE = 2` to draw 2 periodic
copies per axis and check that the structure continues seamlessly across the
cell faces (in 3D this makes the surface 8 times larger, so raise `STEP` too).

In [ ]:
PHASE = None  # True, False, or None for the minority phase
STEP = 1      # marching-cubes step size (2 or 3 for N >= 128)
TILE = 1      # periodic copies per axis (2 shows how the structure repeats)
CELL_COLOR = "#e0461f"  # unit-cell outlines


def tiled(img):
    return np.tile(img, (TILE,) * img.ndim)


def cell_segments(dim):
    """End points of the unit-cell edges of a TILE**dim block, in cell units."""
    ticks = range(TILE + 1)
    segments = []
    for axis in range(dim):
        others = [a for a in range(dim) if a != axis]
        for fixed in itertools.product(ticks, repeat=dim - 1):
            a, b = np.zeros(dim), np.zeros(dim)
            a[others], b[others] = fixed, fixed
            b[axis] = TILE
            segments.append((a, b))
    return segments


def outline_2d(ax):
    for a, b in cell_segments(2):
        ax.plot(*zip(a, b), color=CELL_COLOR, lw=1.5)


def show_2d(img):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(tiled(img).T, cmap="gray", origin="lower", interpolation="nearest",
              extent=(0, TILE, 0, TILE))
    outline_2d(ax)
    ax.set_title(f"{type(generator).__name__}, seed {SEED}  (white = True)")
    ax.axis("off")
    plt.show()


def surface(volume, phase, step):
    """Triangle mesh of one phase, in unit-cell coordinates. The volume is padded
    with the other phase so the solids are closed where they cross the cell faces."""
    solid = tiled(volume if phase else ~volume)
    padded = np.pad(solid, 1, constant_values=False).astype(float)
    verts, faces, _, _ = measure.marching_cubes(padded, 0.5, step_size=step)
    return (verts - 1) / np.array(volume.shape), faces


def show_3d_interactive(verts, faces, phase):
    x, y, z = verts.T
    i, j, k = faces.T
    mesh = go.Mesh3d(
        x=x, y=y, z=z, i=i, j=j, k=k,
        color="#3b6fd8",
        lighting=dict(ambient=0.5, diffuse=0.8, specular=0.1, roughness=0.6),
        hoverinfo="skip",
    )
    lines = np.array([p for a, b in cell_segments(3) for p in (a, b, [np.nan] * 3)])
    cells = go.Scatter3d(x=lines[:, 0], y=lines[:, 1], z=lines[:, 2], mode="lines",
                         line=dict(color=CELL_COLOR, width=4), hoverinfo="skip")
    axis = dict(range=[-0.01, TILE + 0.01], visible=False)
    fig = go.Figure([mesh, cells])
    fig.update_layout(
        title=f"{type(generator).__name__}, seed {SEED}: phase {phase} "
        f"(fraction {np.mean(structure == phase):.3f})",
        scene=dict(xaxis=axis, yaxis=axis, zaxis=axis, aspectmode="cube",
                   camera=dict(eye=dict(x=1.6, y=1.4, z=1.1))),
        width=700, height=650, margin=dict(l=0, r=0, t=50, b=0),
        dragmode="orbit", showlegend=False,
    )
    fig.show()


def show_3d_static(verts, faces, phase):
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(projection="3d")
    ax.plot_trisurf(verts[:, 0], verts[:, 1], faces, verts[:, 2],
                    color="tab:blue", linewidth=0, antialiased=False, shade=True)
    for a, b in cell_segments(3):
        ax.plot(*zip(a, b), color=CELL_COLOR, lw=1.5)
    ax.set(xlim=(0, TILE), ylim=(0, TILE), zlim=(0, TILE), xticks=[], yticks=[], zticks=[])
    ax.set_box_aspect((1, 1, 1))
    ax.set_title(f"{type(generator).__name__}, seed {SEED}: phase {phase}")
    plt.show()


def show_slices(volume):
    mid = [s // 2 for s in volume.shape]
    slices = {"x = 1/2": volume[mid[0]], "y = 1/2": volume[:, mid[1]], "z = 1/2": volume[:, :, mid[2]]}
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
    for ax, (name, sl) in zip(axes, slices.items()):
        ax.imshow(tiled(sl).T, cmap="gray", origin="lower", interpolation="nearest",
                  extent=(0, TILE, 0, TILE))
        outline_2d(ax)
        ax.set_title(name)
        ax.axis("off")
    fig.suptitle("mid-plane slices (white = True)")
    plt.show()


if structure.ndim == 2:
    show_2d(structure)
else:
    phase = (structure.mean() <= 0.5) if PHASE is None else PHASE
    verts, faces = surface(structure, phase, STEP)
    print(f"surface: {len(faces):,} triangles")
    if go is not None:
        show_3d_interactive(verts, faces, phase)
    else:
        show_3d_static(verts, faces, phase)
    show_slices(structure)